# Fine-Tuning PaddleOCR for Egyptian License Plates

Fine-tunes the Arabic PP-OCRv3 recognition model on 2,095 real Egyptian plate images —
dirty, blurry, and degraded — so PaddleOCR learns the specific character patterns it
will encounter in production instead of hallucinating or missing digits.

**Pipeline:**
1. Environment Setup
2. Dataset Structure
3. Character Dictionary
4. Configuration File
5. Download Pretrained Weights
6. Fine-Tune
7. Evaluate
8. Export & Inference

## Configuration

**Edit `DATASET_ROOT` before running anything else.**
Everything else is auto-configured based on whether you're on Kaggle or running locally.

In [ ]:
import os, sys, math

# ── EDIT THIS ────────────────────────────────────────────────────────────────
# Path to the dataset root — must contain train/, valid/, test/ subfolders
# and _annotations.coco.json inside each.
#
# Kaggle example:
#   DATASET_ROOT = "/kaggle/input/egyptian-license-plates/Egyptian License Plate Dataset.v1i.coco"
# Local example:
#   DATASET_ROOT = r"D:/NGU/.../Egyptian License Plate Dataset.v1i.coco"
DATASET_ROOT = "/kaggle/input/egyptian-license-plates/Egyptian License Plate Dataset.v1i.coco"
# ─────────────────────────────────────────────────────────────────────────────

IS_KAGGLE      = os.path.exists("/kaggle/input")
WORK_DIR       = "/kaggle/working" if IS_KAGGLE else os.path.abspath(".")
PADDLEOCR_REPO = os.path.join(WORK_DIR, "PaddleOCR")
PRETRAINED_DIR = os.path.join(WORK_DIR, "pretrain_models")
DICT_PATH      = os.path.join(WORK_DIR, "arabic_plate_dict.txt")
CONFIG_PATH    = os.path.join(WORK_DIR, "arabic_plate_rec.yml")
OUTPUT_DIR     = os.path.join(WORK_DIR, "output", "arabic_plate_rec")
INFERENCE_DIR  = os.path.join(WORK_DIR, "output", "arabic_plate_rec_inference")

for d in [PRETRAINED_DIR, OUTPUT_DIR, INFERENCE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Kaggle  : {IS_KAGGLE}")
print(f"Dataset : {DATASET_ROOT}")
print(f"Work    : {WORK_DIR}")
print(f"Output  : {OUTPUT_DIR}")

## Step 1 — Environment Setup

Clone the PaddleOCR source repository (release/2.6) for the training scripts
(`tools/train.py`, `tools/eval.py`, `tools/export_model.py`).
The pip package only provides inference — training requires the source.

In [ ]:
import subprocess

# Install PaddlePaddle GPU.
# We avoid the cu130 pip index — it pulls cuda-python>=13.0 which conflicts with
# Kaggle's pre-installed RAPIDS stack (which requires cuda-python<13.0).
# The official PaddlePaddle wheel index ships the right CUDA runtime internally.
try:
    import paddle
    print(f"PaddlePaddle already installed: {paddle.__version__}")
except ModuleNotFoundError:
    print("Installing PaddlePaddle GPU (CUDA 12.0 wheel) ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "paddlepaddle-gpu==2.6.2.post120",
         "-f", "https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html",
         "-q"],
        check=True
    )
    import paddle
    print(f"PaddlePaddle installed: {paddle.__version__}")

print(f"CUDA available: {paddle.is_compiled_with_cuda()}")

# Install paddleocr pip package (inference API + dependencies)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "paddleocr==2.6.1", "-q"]
)

# Clone PaddleOCR source (depth=1 for speed — we only need the latest commit)
if not os.path.exists(PADDLEOCR_REPO):
    print("Cloning PaddleOCR release/2.6 ...")
    subprocess.run([
        "git", "clone",
        "--branch", "release/2.6",
        "--depth", "1",
        "https://github.com/PaddlePaddle/PaddleOCR.git",
        PADDLEOCR_REPO
    ], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "-r", os.path.join(PADDLEOCR_REPO, "requirements.txt"), "-q"]
    )
    print("Done.")
else:
    print("PaddleOCR already cloned.")


## Step 2 — Dataset Structure

Run `prepare_dataset.py` to convert the Roboflow COCO JSON annotations into
PaddleOCR's recognition format: `image_path<TAB>plate_text` — one line per image.

Expected output:
```
train_labels.txt  — ~1,557 lines
valid_labels.txt  — ~  354 lines
test_labels.txt   — ~  184 lines
```

> **Kaggle users:** Upload `prepare_dataset.py` alongside this notebook, or add it as a dataset input.

In [ ]:
# Locate prepare_dataset.py — same directory as this notebook
notebook_dir   = os.path.dirname(os.path.abspath("__file__"))
prepare_script = os.path.join(notebook_dir, "prepare_dataset.py")

# Check if label files already exist (skip if already prepared)
train_labels = os.path.join(DATASET_ROOT, "train_labels.txt")
labels_exist = os.path.exists(train_labels)

if not labels_exist:
    if not os.path.exists(prepare_script):
        raise FileNotFoundError(
            f"prepare_dataset.py not found at {prepare_script}.\n"
            "Upload it alongside this notebook and re-run."
        )
    print("Running prepare_dataset.py ...")
    result = subprocess.run(
        [sys.executable, prepare_script, DATASET_ROOT],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("prepare_dataset.py failed.")
else:
    print("Label files already exist — skipping prepare_dataset.py")

# Verify and show samples
print()
for split in ["train", "valid", "test"]:
    path = os.path.join(DATASET_ROOT, f"{split}_labels.txt")
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            lines = [l for l in f if l.strip()]
        print(f"{split}_labels.txt: {len(lines):,} samples")
        if split == "train":
            for line in lines[:3]:
                img, text = line.strip().split("\t", 1)
                print(f"  {os.path.basename(img)}  →  {text}")
    else:
        print(f"WARNING: {path} not found")

## Step 3 — Character Dictionary

PaddleOCR's decoder maps output indices to characters using `dict.txt` — one character
per line. If a character isn't in this file, the model can never predict it.

We build it from the actual labels so it contains exactly the characters that appear
in the dataset — no more, no less.

In [ ]:
chars = set()

for split in ["train", "valid", "test"]:
    path = os.path.join(DATASET_ROOT, f"{split}_labels.txt")
    if not os.path.exists(path):
        continue
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) == 2:
                chars.update(parts[1])

# Sort: Arabic letters first (Unicode order), then Arabic-Indic digits ٠-٩
letters = sorted(c for c in chars if not ('\u0660' <= c <= '\u0669'))
digits  = sorted(c for c in chars if '\u0660' <= c <= '\u0669')
sorted_chars = letters + digits

with open(DICT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(sorted_chars))

print(f"dict.txt written: {len(sorted_chars)} characters")
print(f"  Letters ({len(letters)}): {''.join(letters)}")
print(f"  Digits  ({len(digits)}):  {''.join(digits)}")
print(f"  Path: {DICT_PATH}")

## Step 4 — Configuration File

We use the Arabic PP-OCRv3 architecture (SVTR_LCNet with CTC + NRTR multi-loss)
and tune key hyperparameters for fine-tuning:

| Parameter | Value | Reason |
|-----------|-------|--------|
| `learning_rate` | `0.0001` | Lower than base `0.0005` — adapting, not learning from scratch |
| `max_text_length` | `12` | Egyptian plates have ≤8 chars; 12 gives safe headroom |
| `epoch_num` | `100` | Monitor valid/ acc — stop early if it plateaus for 10+ epochs |
| `character_dict_path` | our `dict.txt` | Only the characters that appear on Egyptian plates |
| `batch_size_per_card` | `64` | Conservative; increase to 128 if GPU memory allows |
| `num_workers` | `4` on Kaggle, `0` on Windows | Windows multiprocessing in DataLoader doesn't work with 0 |

The `RecAug` transform in the training pipeline applies random noise, blur, and
distortion — exactly what we need to teach the model to handle dirty plates.

In [ ]:
BATCH_SIZE   = 64
NUM_WORKERS  = 4 if IS_KAGGLE else 0
PRETRAINED   = os.path.join(PRETRAINED_DIR, "arabic_PP-OCRv3_rec_train", "best_accuracy")

# Compute steps per epoch so we evaluate exactly once per epoch
with open(os.path.join(DATASET_ROOT, "train_labels.txt"), encoding="utf-8") as f:
    n_train = sum(1 for l in f if l.strip())
steps_per_epoch = max(1, math.ceil(n_train / BATCH_SIZE))

TRAIN_LABELS = os.path.join(DATASET_ROOT, "train_labels.txt")
VALID_LABELS = os.path.join(DATASET_ROOT, "valid_labels.txt")
TEST_LABELS  = os.path.join(DATASET_ROOT, "test_labels.txt")

config_yaml = f"""Global:
  use_gpu: true
  epoch_num: 100
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: {OUTPUT_DIR}
  save_epoch_step: 10
  eval_batch_step: [0, {steps_per_epoch}]
  cal_metric_during_train: true
  pretrained_model: {PRETRAINED}
  checkpoints:
  save_inference_dir:
  use_visualdl: false
  infer_img:
  character_dict_path: {DICT_PATH}
  max_text_length: 12
  infer_mode: false
  use_space_char: false
  distributed: false
  save_res_path: {OUTPUT_DIR}/predicts.txt

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0001
    warmup_epoch: 5
  regularizer:
    name: 'L2'
    factor: 3.0e-05

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform:
  Backbone:
    name: MobileNetV1Enhance
    scale: 0.5
    last_conv_stride: [1, 2]
    last_pool_type: avg
    last_pool_kernel_size: [2, 4]
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 64
            depth: 2
            hidden_dims: 120
            use_guide: True
          Head:
            fc_decay: 0.00001
      - NRTRHead:
          nrtr_dim: 96
          max_text_length: 12

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - NRTRLoss:

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc
  ignore_space: false

Train:
  dataset:
    name: SimpleDataSet
    data_dir: {DATASET_ROOT}
    ext_op_transform_idx: 1
    label_file_list:
      - {TRAIN_LABELS}
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - RecAug:
      - MultiLabelEncode:
          gtc_encode: NRTRLabelEncode
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys: [image, label_ctc, label_gtc, length, valid_ratio]
  loader:
    shuffle: true
    batch_size_per_card: {BATCH_SIZE}
    drop_last: true
    num_workers: {NUM_WORKERS}

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: {DATASET_ROOT}
    label_file_list:
      - {VALID_LABELS}
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - MultiLabelEncode:
          gtc_encode: NRTRLabelEncode
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys: [image, label_ctc, label_gtc, length, valid_ratio]
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: {BATCH_SIZE}
    num_workers: {NUM_WORKERS}
"""

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    f.write(config_yaml)

print(f"Config written to: {CONFIG_PATH}")
print(f"  Training samples : {n_train:,}")
print(f"  Steps per epoch  : {steps_per_epoch}")
print(f"  Batch size       : {BATCH_SIZE}")
print(f"  Eval frequency   : every {steps_per_epoch} steps (once per epoch)")

## Step 5 — Download Pretrained Weights

We start from the Arabic PP-OCRv3 weights — already trained on Arabic script — so the
model begins with knowledge of Arabic character shapes and only needs to adapt to
the specific appearance of Egyptian license plates.

Without pretraining we'd be training from random weights — much slower convergence
and much lower final accuracy given only 1,557 training images.

In [ ]:
import urllib.request, tarfile

WEIGHTS_URL = "https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/arabic_PP-OCRv3_rec_train.tar"
TAR_PATH    = os.path.join(PRETRAINED_DIR, "arabic_PP-OCRv3_rec_train.tar")
WEIGHTS_DIR = os.path.join(PRETRAINED_DIR, "arabic_PP-OCRv3_rec_train")

if not os.path.exists(WEIGHTS_DIR):
    print("Downloading pretrained Arabic PP-OCRv3 weights (~9 MB) ...")
    urllib.request.urlretrieve(WEIGHTS_URL, TAR_PATH)
    with tarfile.open(TAR_PATH) as tar:
        tar.extractall(PRETRAINED_DIR)
    os.remove(TAR_PATH)
    print("Done.")
else:
    print("Weights already downloaded.")

print("Contents:", os.listdir(WEIGHTS_DIR))

## Step 6 — Fine-Tune

Runs PaddleOCR's training script. After every epoch it evaluates on `valid/` and saves
a `best_accuracy` checkpoint whenever the score improves — so even if you train too
long, the best model is always preserved.

**What to watch:**

| Metric | Meaning |
|--------|--------|
| `acc` | % of plates where the entire predicted text exactly matches the label |
| `norm_edit_distance` | Softer score — character-level similarity (1.0 = perfect) |

**When to stop early:**  
If `valid/ acc` hasn't improved for 10+ consecutive epochs, training is done.
The `best_accuracy` checkpoint is already saved — no need to wait for epoch 100.

In [ ]:
result = subprocess.run(
    [sys.executable, "tools/train.py", "-c", CONFIG_PATH],
    cwd=PADDLEOCR_REPO
)

if result.returncode != 0:
    print("Training failed — check the output above for errors.")
else:
    best = os.path.join(OUTPUT_DIR, "best_accuracy.pdparams")
    if os.path.exists(best):
        print(f"Training complete. Best checkpoint: {OUTPUT_DIR}/best_accuracy")
    else:
        print(f"Training complete. Checkpoints saved to: {OUTPUT_DIR}")

## Step 7 — Evaluate

Runs the best checkpoint on the held-out `test/` split — 184 plates the model has
never seen during training or any hyperparameter decision.

This is the honest final score. The `valid/` score seen during training was used to
select the best checkpoint, so it's no longer a fully unbiased estimate.

In [ ]:
import yaml

# Write a test-specific config pointing eval dataset to test_labels.txt
BEST_CHECKPOINT  = os.path.join(OUTPUT_DIR, "best_accuracy")
TEST_CONFIG_PATH = os.path.join(WORK_DIR, "arabic_plate_rec_test.yml")

with open(CONFIG_PATH, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["Global"]["checkpoints"]                   = BEST_CHECKPOINT
cfg["Eval"]["dataset"]["label_file_list"]       = [TEST_LABELS]

with open(TEST_CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print("Running evaluation on test split ...")
subprocess.run(
    [sys.executable, "tools/eval.py", "-c", TEST_CONFIG_PATH],
    cwd=PADDLEOCR_REPO
)

## Step 8 — Export & Inference

The training checkpoint (`.pdparams`) must be exported to PaddleOCR's inference format
(`.pdmodel` + `.pdiparams`) before it can be used with the Python API.

After export, the model can be swapped into `PlateReader.py` by pointing
`rec_model_dir` to the inference folder and `rec_char_dict_path` to `dict.txt`.

In [ ]:
# Export best training checkpoint → inference format
print("Exporting model ...")
subprocess.run(
    [
        sys.executable, "tools/export_model.py",
        "-c", CONFIG_PATH,
        "-o",
        f"Global.pretrained_model={BEST_CHECKPOINT}",
        f"Global.save_inference_dir={INFERENCE_DIR}",
        f"Global.character_dict_path={DICT_PATH}",
    ],
    cwd=PADDLEOCR_REPO, check=True
)

print(f"Exported to: {INFERENCE_DIR}")
print("Files:", os.listdir(INFERENCE_DIR))

In [ ]:
# Run inference on a test image
import glob
from paddleocr import PaddleOCR

test_images = glob.glob(os.path.join(DATASET_ROOT, "test", "*.jpg"))
if not test_images:
    test_images = glob.glob(os.path.join(DATASET_ROOT, "test", "*.png"))

if not test_images:
    print("No test images found.")
else:
    test_img = test_images[0]
    print(f"Image: {os.path.basename(test_img)}")

    ocr = PaddleOCR(
        rec_model_dir=INFERENCE_DIR,
        rec_char_dict_path=DICT_PATH,
        use_angle_cls=False,
        use_gpu=True,
        det=False,   # plate crop is already extracted — skip detection
    )

    result = ocr.ocr(test_img, det=False, cls=False)
    if result and result[0]:
        for text, confidence in result[0]:
            print(f"Predicted : {text}  (confidence: {confidence:.3f})")
    else:
        print("No text detected.")

    # Show ground truth
    with open(TEST_LABELS, encoding="utf-8") as f:
        gt = {}
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) == 2:
                gt[parts[0]] = parts[1]

    key = "test/" + os.path.basename(test_img)
    if key in gt:
        print(f"Ground truth: {gt[key]}")

## Using the Fine-Tuned Model in Production

In `PlateReader.py`, replace the `PaddleOCR(...)` constructor with:

```python
self.ocr_ar = PaddleOCR(
    rec_model_dir="/path/to/arabic_plate_rec_inference",
    rec_char_dict_path="/path/to/arabic_plate_dict.txt",
    use_angle_cls=False,
    use_gpu=True,
    det=False,
)
```

The rest of `PlateReader.py` — the 30% header filter, the 2x upscale retry, the
confidence scoring — remains unchanged.